# CHB-MIT 单通道预测 · 纯学生 α=1 基线（复现 261114 的 α=1）

无 teacher、无注意力融合。只训六个单电极学生（纯 BCE，等价于 261114 的 α=1），
最后用**概率平均集成**（六路概率取平均）作为唯一可部署基线。

口径与 261114 α=1 逐条一致：学生输入标准化、折由 `teacher_from_student` 的 keep 决定、
学生 `manual_seed(SEED*1000+fold*10+e)`、全局 `randperm` 打乱、阈值 0.5。

- 代码：`/content/drive/MyDrive/colab`
- EDF：`/content/chbmit`（跑完一个患者删一个）
- 结果：`/content/drive/MyDrive/colab/results_baseline/{pid}/`

In [ ]:
!pip install -q mne

In [ ]:
import gc
import json
import shutil
import sys
from datetime import datetime
from pathlib import Path

from google.colab import drive
drive.mount("/content/drive")

CODE = Path("/content/drive/MyDrive/colab")
DATA = Path("/content/chbmit")
OUT = CODE / "results_baseline"

sys.path.insert(0, str(CODE))
DATA.mkdir(exist_ok=True)
OUT.mkdir(exist_ok=True)

import numpy as np
import torch
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score

import tools as t
import colab_tools as ct
from models import ConvBILSTMClassify

print("device =", t.DEVICE)

In [ ]:
ELECS = ["Fp1-F7", "Fp2-F8", "T7-P7", "T8-P8", "P7-O1", "P8-O2"]
PATIENTS = ["chb%02d" % i for i in range(1, 25) if i not in (4, 15)]

SPH, SOP = 2 * 60, 30 * 60           # 发作前留白 2 min，发作前期 30 min
INTER_THR = 2 * 60 * 60              # 发作间期离发作至少 2 h
WIN_LEN, WIN_OL = 4, 2               # 4 s 窗口，50% 重叠
ENC_VAL, CLS_VAL = 0.1, 0.1
SEED = 0

LR_S, WD_S = 1e-4, 1e-3
EPOCHS_S, PATIENCE_S = 30, 10
BATCH = 32
HIDDEN, DROPOUT = 20, 0.7
CHAN_ATTEN, TEMP_ATTEN = False, True

CONFIG = dict(
    patients=PATIENTS, seed=SEED, sph=SPH, sop=SOP, interictal_thr=INTER_THR,
    window_length=WIN_LEN, window_overlap=WIN_OL, enc_val=ENC_VAL, cls_val=CLS_VAL,
    learning_rate=LR_S, weight_decay=WD_S, epochs=EPOCHS_S, patience=PATIENCE_S,
    batch_size=BATCH, hidden_size=HIDDEN, dropout=DROPOUT,
    channal_attention=CHAN_ATTEN, temporal_attention=TEMP_ATTEN,
    alpha=1.0, threshold=0.5, normalize=True,
)

In [ ]:
torch.backends.cudnn.benchmark = True   # 固定形状，让 cudnn 选最快的卷积算法


def make_batches(*ts, shuffle=False, batch_size=32):
    """张量已经在 GPU 上，直接按索引切片；全局 randperm，与 261114 的 RNG 流一致"""
    n = len(ts[0])
    order = torch.randperm(n, device=ts[0].device) if shuffle else torch.arange(n, device=ts[0].device)
    for i in range(0, n, batch_size):
        idx = order[i:i + batch_size]
        yield tuple(x[idx] for x in ts)


def predict(model, x, y, batch_size=BATCH):
    probs, targets = [], []
    model.eval()
    with torch.no_grad():
        for bx, by in make_batches(x, y, batch_size=batch_size):
            probs.append(torch.sigmoid(model(bx)).ravel())
            targets.append(by.ravel())
    return torch.cat(probs).cpu().numpy(), torch.cat(targets).cpu().numpy()

In [ ]:
for pid in PATIENTS:
    done = OUT / pid / ("%s.json" % pid)
    if done.exists():
        print("[跳过] 已完成", pid)
        continue
    print("=" * 15, pid, "=" * 15)
    out = OUT / pid
    out.mkdir(parents=True, exist_ok=True)

    # 1. 下载 + 预处理
    pdir = ct.download_patient(pid, DATA)
    windows, meta = t.preprocess(str(pdir), SPH, SOP, INTER_THR, WIN_LEN, WIN_OL, ELECS)

    # 2. 学生折（不丢折）→ 借 teacher_from_student 的 keep 决定折的选取（与 261114 一致）
    student_ds = ct.student_dataset(ELECS, meta, SEED, (ENC_VAL, CLS_VAL))
    teacher_ds = t.teacher_from_student(ELECS, meta, windows, student_ds[0], SEED)
    keep = {d["seizure_id"] for d in teacher_ds}
    del teacher_ds
    student_ds = [[f for f in elec if f["seizure_id"] in keep] for elec in student_ds]

    # 3. 每折归一化参数：只传学生折（教师折的排除是学生折的子集，结果不变）
    norm_stats = [t.fold_norm_stats(meta, windows, ELECS, s) for s in student_ds[0]]
    print("[%s] %d 折 %s" % (pid, len(student_ds[0]), [f["seizure_id"] for f in student_ds[0]]))

    data = {"config": CONFIG, "test_index": []}

    for fold, s0 in enumerate(student_ds[0]):
        mean, std = norm_stats[fold]
        print("---- fold %d/%d seizure %s ----" % (fold + 1, len(student_ds[0]), s0["seizure_id"]))

        # 4. 学生输入：标准化后放 GPU（六个电极各自按 mean[e:e+1]/std[e:e+1]）
        sw = []
        for e, elec in enumerate(ELECS):
            s = student_ds[e][fold]
            dd = {}
            for key in ("train", "val", "test"):
                idx = s[key + "_idx"]
                dd[key + "_wins"] = t.norm_wins(
                    torch.tensor(windows[elec][idx][:, None, :], dtype=torch.float32),
                    mean[e:e + 1], std[e:e + 1]).to(t.DEVICE)
                dd["y_" + key] = torch.tensor(meta[elec].iloc[idx].label.to_numpy(),
                                              dtype=torch.float32).unsqueeze(1).to(t.DEVICE)
            sw.append(dd)

        test_probs, elec_metrics = [], []
        tt = None
        for e, elec in enumerate(ELECS):
            s = sw[e]
            torch.manual_seed(SEED * 1000 + fold * 10 + e)   # 与 261114 α=1 同一种子
            student = ConvBILSTMClassify(channal_attention=CHAN_ATTEN, temporal_attention=TEMP_ATTEN,
                                         dropout=DROPOUT, hidden_size=HIDDEN).to(t.DEVICE)
            opt_s = torch.optim.Adam(student.parameters(), lr=LR_S, weight_decay=WD_S)
            best_val, best_state, counter = float("inf"), {}, 0
            for epoch in range(EPOCHS_S):
                student.train()
                for x, y in make_batches(s["train_wins"], s["y_train"], shuffle=True, batch_size=BATCH):
                    opt_s.zero_grad()
                    loss = F.binary_cross_entropy_with_logits(student(x).squeeze(-1), y.squeeze(-1))
                    loss.backward()
                    opt_s.step()
                student.eval()
                val_loss, nb = 0, 0
                with torch.no_grad():
                    for x, y in make_batches(s["val_wins"], s["y_val"], batch_size=BATCH):
                        val_loss += F.binary_cross_entropy_with_logits(student(x).squeeze(-1), y.squeeze(-1)).item()
                        nb += 1
                val_loss /= nb
                if val_loss < best_val:
                    best_val, counter = val_loss, 0
                    best_state = {k: v.cpu().clone() for k, v in student.state_dict().items()}
                else:
                    counter += 1
                    if counter >= PATIENCE_S:
                        break
            student.load_state_dict(best_state)
            tp, tt = predict(student, s["test_wins"], s["y_test"])
            test_probs.append(tp)
            elec_metrics.append(dict(elec=elec, **ct.classification_metrics(tt, tp)))
            del student, opt_s

        # 5. 概率平均集成（唯一保留的融合，最终基线）
        tp_mean = np.mean(test_probs, axis=0)
        m_pmean = ct.classification_metrics(tt, tp_mean)
        data["test_index"].append(dict(fold=fold, seizure_id=int(s0["seizure_id"]), alpha=1.0,
                                       metrics=elec_metrics, metrics_pmean=[m_pmean]))
        print("  fold %d | 概率平均 AUC %.4f ACC %.4f SEN %.4f SPE %.4f"
              % (fold + 1, m_pmean["auc"], m_pmean["accuracy"],
                 m_pmean["sensitivity"], m_pmean["specificity"]))

        del sw
        torch.cuda.empty_cache()
        gc.collect()

    with open(out / ("%s.json" % pid), "w", encoding="utf-8") as f:
        json.dump(data, f, indent=4, ensure_ascii=False)
    del student_ds, norm_stats, windows, meta, data
    torch.cuda.empty_cache()
    gc.collect()
    ct.clear_patient(pid, DATA)
    print("[完成]", pid)

## 汇总

In [ ]:
df = ct.summarize_distill(OUT)
print(df[["auc_pmean", "accuracy_pmean", "sensitivity_pmean", "specificity_pmean"]].describe())
df.groupby("patient")[["auc_pmean"]].mean().sort_values("auc_pmean")